In [4]:
%pip install xgboost

"""
================================================================================
MODULE B: TIME-SERIES DRIFT PREDICTOR (XGBoost Regression)
================================================================================
GOAL (from the problem statement):
    Take a component's parametric reading at 0 hours and 24 hours of burn-in
    testing, and PREDICT what its reading will be at 168 hours.
    A big predicted 168h value means the part is drifting badly and should be
    flagged for rejection.

WHY XGBoost INSTEAD OF LINEAR REGRESSION?
    Linear regression fits ONE straight line/plane through the data:
        y = w1*x1 + w2*x2 + b
    That only works well if the relationship between the inputs and the
    output is a straight line. Drift is NOT a straight line - a healthy part
    barely changes, but a defective part can suddenly shoot up 100x. XGBoost
    instead builds many small decision trees, one after another, where every
    new tree tries to correct the mistakes of the trees before it. Stacking
    many simple trees lets it bend and curve to fit patterns a straight line
    cannot.

WHAT "UNDERFITTING" AND "OVERFITTING" MEAN HERE:
    - Underfitting: the model is too simple / stopped too early. It performs
      badly on BOTH the data it trained on and new data. (Like drawing a
      straight line through a clearly curved pattern.)
    - Overfitting: the model is too complex / trained too long. It performs
      GREAT on the data it has already seen (training data) but badly on new
      data (test data), because it memorized noise instead of learning the
      real pattern.
    - The goal (a "good fit") is when training and test performance are close
      to each other AND both are reasonably low. We check this below with
      numbers AND with graphs.
================================================================================
"""


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/48.9 MB 6.4 MB/s eta 0:00:08
   - -------------------------------------- 2.1/48.9 MB 4.8 MB/s eta 0:00:10
   -- ------------------------------------- 3.1/48.9 MB 4.5 MB/s eta 0:00:11
   -- ------------------------------------- 3.7/48.9 MB 4.1 MB/s eta 0:00:12
   --- ------------------------------------ 4.5/48.9 MB 4.3 MB/s eta 0:00:11
   ---- ----------------------------------- 5.0/48.9 MB 3.9 MB/s eta 0:00:12
   ---- ----------------------------------- 5.5/48.9 MB 3.6 MB/s eta 0:00:13
   ---- ----------------------------------- 6.0/48.9 MB 3.4 MB/s eta 0:00:13
   ----- ---------------------------------- 6.6/48.9 MB 3.3 MB/s eta 0:00:13
   ----- ---------------------------------- 7.1/48.9 MB 3.2 MB/s eta 0:00:13
   ----- ---------------------------------- 7.3/48.9 MB 3.2 MB/s eta 0:00:14
   ------ ---

'\n================================================================================\nMODULE B: TIME-SERIES DRIFT PREDICTOR (XGBoost Regression)\n================================================================================\nGOAL (from the problem statement):\n    Take a component\'s parametric reading at 0 hours and 24 hours of burn-in\n    testing, and PREDICT what its reading will be at 168 hours.\n    A big predicted 168h value means the part is drifting badly and should be\n    flagged for rejection.\n\nWHY XGBoost INSTEAD OF LINEAR REGRESSION?\n    Linear regression fits ONE straight line/plane through the data:\n        y = w1*x1 + w2*x2 + b\n    That only works well if the relationship between the inputs and the\n    output is a straight line. Drift is NOT a straight line - a healthy part\n    barely changes, but a defective part can suddenly shoot up 100x. XGBoost\n    instead builds many small decision trees, one after another, where every\n    new tree tries to correct the

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [7]:
# ------------------------------------------------------------------------
# STEP 0: SETTINGS YOU CAN CHANGE
# ------------------------------------------------------------------------
# Keep every "knob" a beginner might want to tweak in one place, at the top.

DATA_PATH = "isro_burnin_ess_synthetic_dataset_csv.xlsx"

# The dataset has 3 parametric signals, each measured at 0h/24h/96h/168h:
#   "iddq_uA"               (standby current)
#   "leakage_current_nA"    (leakage current)
#   "propagation_delay_ns"  (propagation delay)
# Module B only needs ONE of them at a time. Change this string to switch
# which signal the model predicts - the rest of the code does not need to
# change.
PARAMETER = "iddq_uA"

# XGBoost hyperparameters. Kept modest on purpose: a small learning rate,
# a shallow tree depth, and a large-but-early-stopped number of trees are
# the standard recipe for avoiding overfitting.
MODEL_SETTINGS = dict(
    n_estimators=1000,       # max number of trees to try (early stopping will cut this short)
    max_depth=4,             # how deep each tree can grow (small = simpler = less overfitting)
    learning_rate=0.05,      # how much each new tree is allowed to correct the previous ones
    subsample=0.8,           # each tree only sees 80% of the rows (adds randomness -> less overfitting)
    colsample_bytree=0.8,    # each tree only sees 80% of the columns (same reason)
    random_state=42,         # makes the results repeatable
)
EARLY_STOPPING_ROUNDS = 20   # stop training if validation error hasn't improved for this many trees

In [8]:
# ------------------------------------------------------------------------
# STEP 1: LOAD THE DATA
# ------------------------------------------------------------------------
df = pd.read_excel(DATA_PATH)

feature_cols = [f"{PARAMETER}_0h", f"{PARAMETER}_24h"]   # inputs (X)
target_col = f"{PARAMETER}_168h"                          # what we want to predict (y)

# Keep only the columns we need, and drop any row with a missing value.
data = df[feature_cols + [target_col]].dropna().reset_index(drop=True)
print(f"Using {len(data)} rows to predict '{target_col}' from {feature_cols}")

FileNotFoundError: [Errno 2] No such file or directory: 'isro_burnin_ess_synthetic_dataset_csv.xlsx'

In [ ]:
# ------------------------------------------------------------------------
# STEP 2: LOG-TRANSFORM THE VALUES
# ------------------------------------------------------------------------
# In linear regression you use the raw numbers as-is. Here we can't, because
# a healthy part might read "12" while a badly-drifting part reads
# "900,000" - the values span 5+ orders of magnitude. Without this step, the
# model would spend all its effort on the few huge values and largely ignore
# everything else. np.log1p(x) = log(1 + x) squeezes big numbers down while
# barely touching small ones, the same way a log-scale axis does on a graph.
# We undo this later with np.expm1(x) = exp(x) - 1 whenever we want to look
# at values in their original, real-world units.
X = np.log1p(data[feature_cols])
y = np.log1p(data[target_col])

In [ ]:
# ------------------------------------------------------------------------
# STEP 3: SPLIT INTO TRAIN / VALIDATION / TEST
# ------------------------------------------------------------------------
# Train      -> the data the model directly learns from.
# Validation -> data the model never learns from, but we peek at it DURING
#               training to know when to stop (this is how we prevent
#               overfitting - see "early stopping" below).
# Test       -> data the model has never seen or peeked at, used only ONCE
#               at the very end to report the final, trustworthy performance.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Train rows: {len(X_train)} | Validation rows: {len(X_val)} | Test rows: {len(X_test)}")


In [ ]:
 #------------------------------------------------------------------------
# STEP 4: TRAIN THE XGBOOST MODEL (WITH EARLY STOPPING)
# ------------------------------------------------------------------------
# "Early stopping" = after adding each new tree, check the error on the
# VALIDATION set. If it stops improving for EARLY_STOPPING_ROUNDS trees in a
# row, stop adding more trees. This is the main defense against overfitting:
# training error can always keep dropping, but the validation error is our
# honest signal for when the model has started memorizing noise instead of
# learning the real pattern.
model = XGBRegressor(
    **MODEL_SETTINGS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    eval_metric="mae",
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],  # tracked every round for the learning curve below
    verbose=False,
)

print(f"\nTraining stopped at tree number: {model.best_iteration}")

In [ ]:
# ------------------------------------------------------------------------
# STEP 5: CHECK FOR UNDERFITTING / OVERFITTING WITH NUMBERS
# ------------------------------------------------------------------------
# MAE (Mean Absolute Error) = on average, how far off is the prediction, in
# the same units as the target. R2 = fraction of the pattern in the data
# that the model explains (1.0 is a perfect fit, 0.0 is no better than
# always predicting the average).
def report(name, X_split, y_split_log):
    pred_log = model.predict(X_split)
    # undo the log-transform so the error is reported in real, readable units
    pred = np.expm1(pred_log)
    actual = np.expm1(y_split_log)
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(y_split_log, pred_log)
    print(f"{name:>10} -> MAE: {mae:8.3f} | R2: {r2:.3f}")
    return pred, actual

print("\nModel performance (lower MAE is better, R2 closer to 1.0 is better):")
report("Train", X_train, y_train)
report("Validation", X_val, y_val)
test_pred, test_actual = report("Test", X_test, y_test)

print(
    "\nHow to read this: if Train MAE is much lower than Test MAE, the model "
    "is OVERFITTING. If both Train and Test MAE are high, the model is "
    "UNDERFITTING. Here they should be close and both reasonably small."
)

In [ ]:
# ------------------------------------------------------------------------
# STEP 6: GRAPH 1 - LEARNING CURVE (train vs validation error per tree)
# ------------------------------------------------------------------------
# This is the clearest visual check for over/underfitting:
#   - If the train line keeps falling while the validation line rises again,
#     that gap is overfitting.
#   - If both lines flatten out high (barely improve from the start), that
#     is underfitting.
#   - What we want: both lines drop together and flatten out close to each
#     other - which is what early stopping is designed to give us.
results = model.evals_result()
train_curve = results["validation_0"]["mae"]
val_curve = results["validation_1"]["mae"]

plt.figure(figsize=(7, 5))
plt.plot(train_curve, label="Training error")
plt.plot(val_curve, label="Validation error")
plt.axvline(model.best_iteration, color="gray", linestyle="--", label="Early-stopping point")
plt.xlabel("Number of trees added")
plt.ylabel("MAE (log scale of target)")
plt.title(f"Learning Curve - {PARAMETER}")
plt.legend()
plt.tight_layout()
plt.savefig("learning_curve.png", dpi=150)
plt.show()

In [ ]:
# ------------------------------------------------------------------------
# STEP 7: GRAPH 2 - PREDICTED vs ACTUAL (on the untouched test set)
# ------------------------------------------------------------------------
# Every point is one component from the test set. The dashed line is a
# perfect prediction (predicted = actual). Points sitting close to that line
# mean the model's forecast for Value_168h is accurate; points far from it
# are the components the model struggles with.
plt.figure(figsize=(6, 6))
plt.scatter(test_actual, test_pred, alpha=0.4, s=15)
lims = [min(test_actual.min(), test_pred.min()), max(test_actual.max(), test_pred.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xscale("log")
plt.yscale("log")
plt.xlabel(f"Actual {target_col}")
plt.ylabel(f"Predicted {target_col}")
plt.title(f"Predicted vs Actual (Test Set) - {PARAMETER}")
plt.legend()
plt.tight_layout()
plt.savefig("predicted_vs_actual.png", dpi=150)
plt.show()


In [ ]:
# ------------------------------------------------------------------------
# STEP 8: PREDICT ON NEW DATA (example of how to use the trained model)
# ------------------------------------------------------------------------
# This is how someone would use the finished model on a brand new component
# that only has 0h and 24h readings so far.
def predict_168h(value_0h, value_24h):
    """Given a component's 0h and 24h readings, forecast its 168h value."""
    x_new = pd.DataFrame(
        [[np.log1p(value_0h), np.log1p(value_24h)]], columns=feature_cols
    )
    pred_log = model.predict(x_new)[0]
    return np.expm1(pred_log)


example_0h, example_24h = data[feature_cols].iloc[0]
print(
    f"\nExample: a component with {PARAMETER}_0h={example_0h:.2f} and "
    f"{PARAMETER}_24h={example_24h:.2f} is forecast to reach "
    f"{predict_168h(example_0h, example_24h):.2f} at 168h."
)